# Action-space object individuation — does an interaction affordance carve a *grabbable object handle* into the PASSIVE latent?

**Direction:** `research/directions/action-space-object-individuation.md` · worker notebook (standalone; the master
`00_master_editability.ipynb` and the Exp-2 `action_conditioned_structure.ipynb` are **not** modified).

**The reframe (do not lose this).** This is **not** "bigger actions" and **not** editing via the action channel.
The question is object *individuation*: does training on an interaction affordance reorganize the **passive** latent
(action channel OFF = no-op) into a **separable, localizable, grabbable handle that IS "object k"** — a real object
that generalizes to interventions never trained — versus a memorised *button*? Editability is only the **probe**.

**ALL eval is on the passive / no-op latent, using the master §4 editors** — which are a **different write-mechanism**
than the action channel the models were trained on (they bypass the transition dynamics and write the state directly).
So "can a §4 editor grab object k in the passive latent?" is the **interface-generalization** test: did the affordance
move object-hood *into the state* (grabbable by any mechanism) or leave it *in the input→dynamics pathway* (a button)?

**Five passive GRUs (all measured no-op), dataset-4 held-out:**
1. **Baseline** — passive GRU on clean dataset 4 (`runs/gru/7_dset4_gru_400epochs`); no perturbations, no actions.
2. **Perturbed-passive (teleport)** — plain GRU on the **teleport-perturbed** trajectories with the **action channel
   withheld** (`runs/gru/M_teleport_ctrl`); sees only the perturbed obs.
3. **M_dxdy** — continuous-action GRU trained on large relative `(dx,dy)` displacements (`runs/gru/M_dxdy`).
4. **M_teleport** — continuous-action GRU trained on absolute in-frustum placements (`runs/gru/M_teleport`).
5. **M_axis** — continuous-action GRU trained on **x-only** relative displacements (`runs/gru/M_axis`); the
   content-generalization probe (edited along **y**, which it never saw).

**Confound triad** (localises any effect to *action-knowledge*, not perturbation-diversity): Baseline → Perturbed-passive
= perturbation-diversity; **Perturbed-passive → M_teleport = action-knowledge** (the headline gap, same trajectories).


In [ ]:
# [1] Bootstrap: imports, config, dataset-4 eval splits (passive held-out), load all five models.
import sys, os, time, json
sys.path.insert(0, "../../../..")
from dataclasses import replace, asdict
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from tqdm.auto import tqdm
from IPython.display import display, Markdown
import h5py
from torch.utils.data import DataLoader

import pim.eval as ev
from pim.extractors import LinearExtractor, MLPExtractor, StateDefinition
from pim.editors import (probe_decomposition, inject_state, fit_state_subspace,
    project_to_subspace, offmanifold_residual, fit_local_subspace, manifold_steer, gradient_steer)
from pim.editors.manifold_steering import _pca_subspace
from pim.eval.controllability import _rollout
from pim.world_models import load_checkpoint, load_dataset
from pim.world_models.dataloader import ObservationDataset
from pim.world_models.action_gru_continuous import ActionGRUContinuousModel, ActionContinuousModelConfig
from pim.simulator.config import SimConfig
from pim.simulator.actions_continuous import (simulate_with_continuous_actions, denormalize_action,
    norm_consts, generate_continuous_action_dataset)

torch.manual_seed(0); np.random.seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE, NUM_WORKERS, N_OBJ = 512, 6, 2
REPO = "../../../.."
DATA_DIR = f"{REPO}/datasets/4_fixed_refl_inview"     # EVAL (clean, passive, held-out) — SAME for every model
OUT = "/tmp/action_space_object"; os.makedirs(OUT, exist_ok=True)
N_TEST_EVAL = 2500   # test subset for §1-§3 banks (identical subset for every model; bounds probe-fit time)

CKPT = {"Baseline": "7_dset4_gru_400epochs", "Perturbed-passive (teleport)": "M_teleport_ctrl",
        "M_dxdy": "M_dxdy", "M_teleport": "M_teleport", "M_axis": "M_axis"}
ACTION_MODELS = {"M_dxdy": "dxdy", "M_teleport": "teleport", "M_axis": "axis_x"}   # name -> affordance mode

def load_any(run_name):
    """Load a checkpoint: continuous-action GRU if its config has n_obj, else plain GRU via load_checkpoint."""
    path = f"{REPO}/runs/gru/{run_name}/best_model.pt"
    sd = torch.load(path, map_location=DEVICE)
    if "n_obj" in sd["model_config"]:
        m = ActionGRUContinuousModel(ActionContinuousModelConfig(**sd["model_config"])).to(DEVICE).eval()
        m.load_state_dict(sd["model_state"])
        for p in m.parameters(): p.requires_grad_(False)
        return m, sd.get("val_loss", float("nan"))
    m, info = load_checkpoint(path, device=DEVICE)
    return m, info.val_loss

bundle = load_dataset(DATA_DIR, n_obj_keep=N_OBJ)
test, edits = bundle.test, bundle.edits
DT = float(test.config["dataset"]["sim"]["dt"])
NE = min(N_TEST_EVAL, test.n_samples)
sub_loader = DataLoader(ObservationDataset(test.h5_path, np.arange(NE), keys=("obs_intensity",)),
                        batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

RAW = {name: load_any(run) for name, run in CKPT.items()}
H = RAW["Baseline"][0].hidden_size
SIM4 = SimConfig(**{k: test.config["dataset"]["sim"][k] for k in SimConfig.__dataclass_fields__})
print(f"device={DEVICE} | eval = dataset4 test[:{NE}] + edits({edits.n_samples}) | H={H}")
for name in CKPT: print(f"  {name:28s} {CKPT[name]:24s} val_loss {RAW[name][1]:.5f}")
print(f"norm consts (NX,Y_MID,Y_HALF) = {norm_consts(SIM4)}")


In [ ]:
# [2] Physical (pos, vel) targets on the eval subset, aligned to hidden states (positions[:, :-1]).
v_test = h5py.File(test.h5_path, "r")["velocities"][:NE, :, :N_OBJ, :].astype(np.float32)  # (NE,40,2,2)
vel_tf = v_test[:, :-1]                                  # (NE,39,2,2) aligned to states
pos_tf = test.positions[:NE, :-1, :N_OBJ, :]            # (NE,39,2,2)
vis_tf = test.is_visible[:NE, :-1, :N_OBJ].all(axis=2)  # (NE,39) both objects visible
obs_eval = test.obs[:NE]                                 # (NE,40,R) noisy passive obs
posflat_tf = pos_tf.reshape(*pos_tf.shape[:2], N_OBJ * 2)               # [x0,y0,x1,y1]
velflat_tf = vel_tf.reshape(*vel_tf.shape[:2], N_OBJ * 2)               # [vx0,vy0,vx1,vy1]
posvel_tf = np.concatenate([posflat_tf, velflat_tf], -1)               # (NE,39,8)
LATE_T = 15
print(f"eval subset NE={NE} | visible aligned frames {int(vis_tf.sum())} | physical DOF = {N_OBJ*4}")


In [ ]:
# [3] Themes + generic probe / g-fit helpers + a text+markdown table printer (reused across all sections).
OK = {"blue": "#0072B2", "orange": "#D55E00", "green": "#009E73", "pink": "#CC79A7", "yellow": "#E69F00",
      "grey": "#999999", "sky": "#56B4E9", "violet": "#8172B3", "red": "#C1272D"}
def style_ax(ax): ax.spines[["top", "right"]].set_visible(False); ax.grid(alpha=0.25, lw=0.6)
plt.style.use("default")

def _fit_regress(X, Y, kind, hidden=256, n_epochs=100, lr=2e-3, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    Din, Dout = X.shape[1], Y.shape[1]
    Xt = torch.from_numpy(X.astype(np.float32)).to(DEVICE); Yt = torch.from_numpy(Y.astype(np.float32)).to(DEVICE)
    if kind == "linear":
        Xa = torch.cat([Xt, torch.ones(Xt.shape[0], 1, device=DEVICE)], 1)
        sol = torch.linalg.lstsq(Xa, Yt).solution
        with torch.no_grad(): pred = Xa @ sol
    else:
        net = nn.Sequential(nn.Linear(Din, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU(),
                            nn.Linear(hidden, Dout)).to(DEVICE)
        opt = torch.optim.Adam(net.parameters(), lr=lr); bs = 4096; Nn = Xt.shape[0]
        for ep in range(n_epochs):
            perm = torch.randperm(Nn, device=DEVICE)
            for i in range(0, Nn, bs):
                idx = perm[i:i + bs]
                loss = ((net(Xt[idx]) - Yt[idx]) ** 2).mean()
                opt.zero_grad(); loss.backward(); opt.step()
        net.eval()
        with torch.no_grad(): pred = net(Xt)
    resid2 = ((pred - Yt) ** 2).sum(0); tot2 = ((Yt - Yt.mean(0, keepdim=True)) ** 2).sum(0)
    r2_pc = (1 - resid2 / torch.clamp(tot2, min=1e-12)).cpu().numpy()
    r2_all = float(1 - resid2.sum() / tot2.sum())
    resid_frac = float((((pred - Yt) ** 2).sum() / (Yt ** 2).sum()).sqrt())
    return pred.cpu().numpy(), r2_all, r2_pc, resid_frac

def fit_probe(feats_tf, y_tf, mask, kind, **kw):
    X = feats_tf[mask]; Y = y_tf[mask]
    pred, r2, r2pc, rfrac = _fit_regress(X, Y, kind, **kw)
    return dict(r2=r2, r2pc=r2pc, resid_frac=rfrac, n=X.shape[0])

def both_table(data, columns, row_hdr=""):
    """data: {rowname: {key: val}}; columns: [(key, disp, fmt)]. Prints a text table AND displays markdown."""
    w = 26
    def cell(v, f): return "nan" if isinstance(v, float) and np.isnan(v) else (f.format(v) if not isinstance(v, str) else v)
    print(f"{row_hdr:>{w}s}" + "".join(f"{d:>{w}s}" for _, d, _ in columns))
    for rn, vals in data.items():
        print(f"{str(rn):>{w}s}" + "".join(f"{cell(vals[k], f):>{w}s}" for k, _, f in columns))
    lines = ["| " + row_hdr + " | " + " | ".join(d for _, d, _ in columns) + " |",
             "|" + "---|" * (len(columns) + 1)]
    for rn, vals in data.items():
        lines.append("| **" + str(rn) + "** | " + " | ".join(cell(vals[k], f) for k, _, f in columns) + " |")
    display(Markdown("\n".join(lines)))
print("helpers ready")


## Metric definitions (formulas up front)

All probes are fit **in-sample** on the passive dataset-4 test subset (`N_TEST_EVAL` samples, identical subset for
every model). "late-t" = frames t ≥ 15 (converged-filter regime); "early-t" = t < 15. Physical reference = **8 DOF**
(2 objects × (2 pos + 2 vel)). **RMSE** (not MSE), obs intensity ∈ [0,1], throughout. Every §4 metric is computed on the
**passive (no-op) latent** and the §4 editors are a **different write-mechanism** than the trained action channel.

**Datasets & controls (read this).** Eval is always **dataset 4** — the **edits split** for §4 targets, the **test split** for §1–§3 (same subset, every model). But each action model trains on its **own** dataset (`6_cont_dxdy` / `7_cont_teleport` / `8_cont_axis_x`; different action distributions). There is **one** perturbed-passive control, trained on the **teleport** data (channel withheld) — so the clean action-knowledge contrast (perturbed→action) holds **only for teleport**; `M_dxdy`/`M_axis` are compared to the shared `Baseline` only. The content-generalization edits (Fig 6) are **synthesized in-notebook** (each edit object moved 2.5 units purely along x or y from its pre-edit position, then rendered) — they do **not** come from the edits split.

| metric | definition | dir |
|---|---|---|
| hidden state `h` | GRU state after teacher-forcing the passive (no-op) obs to frame t; `h ∈ R^256` | — |
| **§1** PCA hull @p | # PCA components of the visited-`h` bank for ≥ p of the variance | — |
| **§1** intrinsic dim (TwoNN) | model-free `d = 1/mean(log(r₂/r₁))`, r₁,r₂ = 1st/2nd-NN distances (Facco 2017) | — |
| **§1** intrinsic dim (MLE) | Levina–Bickel MLE over k=20 NN, bias-corrected ×(k−2)/(k−1) | — |
| **§1** tangent rotation | mean principal angle (deg) between local-PCA tangents (k=64 NN, top-8) of a state and its NN | — |
| **§2** recoverability R² | `1 − ‖Y − probe(h)‖²/‖Y − Ȳ‖²`, Y ∈ {pos, vel}; linear vs MLP probe | ↑ |
| **§3** fiber residual | `‖h − g(pos,vel)‖ / ‖h‖`, g linear or MLP; 0 = `h` a function of the 8-dim physical state | ↓ |
| **§4** readout RMSE | position RMSE of the linear probe read off the edited state vs the teleport target (state-space) | ↓ |
| **§4** GT next-step RMSE | RMSE(model's next generated obs, true post-edit obs) — observation-space accuracy | ↓ |
| **swap / swap₀** (reference) | the **true-state swap** = teacher-force the model on the *true* post-edit obs → `h_swap` → rollout; `swap₀` = its step-0 obs. It is the **100% denominator** for reach/collateral ("what actually observing the edit does to object k's rays"). A **soft** reference (belief-inertia-limited), NOT a hard ceiling. | — |
| **§4 reach** (% of swap) | `100·RMSE(edited₀, unsteered₀)[obj-k rays] / RMSE(swap₀, unsteered₀)[obj-k rays]`; 100% = a real teleport's effect on object k's rays | ↑ |
| **§4 collateral** (% of swap) | `100·RMSE(edited₀, unsteered₀)[other-object rays] / RMSE(swap₀, unsteered₀)[obj-k rays]`; disturbance of the OTHER object, in the same units as reach; clean handle → 0 | ↓ |
| **§4 ghost ratio** | mean intensity on pre-edit-only (vacated) rays ÷ unsteered; < 1 = object left its old location | ↓ |
| **§4 anti-reversion** | `mean_{s=10..14} RMSE(edited_s, unsteered_s) / RMSE(edited₀, unsteered₀)`; does the change *stay away from the unedited rollout* (≥1) or revert toward it (<1)? Measures **stickiness, NOT correctness** | ↑ |
| **§4 GT-traj RMSE** | `mean_s RMSE(edited_s, GT post-edit_s)` over the K-step rollout (we have the true post-edit obs); direct **fidelity** — did the edit achieve *and hold* the true post-edit world? | ↓ |
| **§4** leave-out local-PCA resid | manifold residency of the edited state (fraction; vs the real-state reference) | — |
| **content-gen ratio** (`M_axis`) | `reach(y-edit) / reach(x-edit)` on the passive latent (readout & obs); ≈1 ⇒ generalises across content ⇒ real object; ≪1 ⇒ memorised affordance | ↑ |

**Object handle** = a passive-latent direction that, when written by a §4 editor targeting object k, moves object k's
rays toward its target (**reach**↑), leaves the other object's rays alone (**collateral**↓), clears object k's old
location (**ghost**↓), and holds over the rollout (**persistence**↑). A clean, selective, ghost-free, persistent handle
= "object k is individuated (grabbable by an untrained mechanism)".

**§4 references** (never editors): **GT (sim)** = the simulator's *time-evolving* clean post-edit obs; **Unsteered** =
rollout from the un-edited state; **True-state swap** = rollout from teacher-forcing the *actual* post-edit observations
(a **soft** reference — belief-inertia-limited, not a hard ceiling: a direct latent write could in principle exceed it). **Editors:** Readout injection, MLP-probe gradient, Global-PCA
projection, PCA geodesic, Decoder-gradient (oracle). Provenance for the geometry/recoverability pipeline: mirrors the
Exp-2 `action_conditioned_structure.ipynb` and master `00_master_editability.ipynb` §1–§4.


## Exposition — the affordances are perceptually LARGE (the independent variable)

Before the metrics, so the action spaces are legible. **(E1)** for each affordance type, the mean per-event `|Δobs|`
between the acted (moved) world and the base (un-moved) world — confirming the moves are **much larger** than Exp-2's
0.7-unit nudge (Exp-2 reported change-action `|Δobs|` ≈ 0.03–0.13). **(E2)** a *change-the-action* sanity check per
action-cond model — same input, feed no-op vs a real action → the predicted rollout diverges → the action channel is
causally used. Demo obs are rendered **clean** (`obs_noise_std=0`) for legibility; every model was trained on **noisy**
obs (0.2). Move magnitude ≈ uniform in ±4 world units (relative) / anywhere in-frustum (teleport).

In [ ]:
# [E1] Perceptual magnitude of each affordance: mean per-event |Δobs| (moved world vs base world), clean renders.
from dataclasses import replace as _dc_replace
from pim.simulator.sim import simulate as _simulate, Scene as _Scene
from pim.simulator.renderer import render_scene as _render
cfg_clean = _dc_replace(SIM4, obs_noise_std=0.0)
EXPO_N, MODE_COLOR = 150, {"dxdy": OK["green"], "teleport": OK["orange"], "axis_x": OK["violet"]}
expo = {}
for mode in ["dxdy", "teleport", "axis_x"]:
    dobs_evt, mags = [], []
    for sd in range(EXPO_N):
        cfg = _dc_replace(cfg_clean, seed=sd)
        base = _simulate(cfg); _, _, obs_base = _render(base)
        moved, acts = simulate_with_continuous_actions(cfg, mode=mode, move_scale=4.0, p_action=0.30, action_seed=sd + 8_000_000)
        _, _, obs_moved = _render(moved)
        for (s, o) in np.argwhere(acts[:, :, 0] > 0.5):
            dobs_evt.append(float(np.abs(obs_moved[s + 1] - obs_base[s + 1]).mean()))   # persistent-move effect at s+1
            a1, a2 = denormalize_action(mode, acts[s, o, 1], acts[s, o, 2], SIM4)
            mags.append(float(np.linalg.norm(moved.positions[s + 1, o] - moved.positions[s, o])))
    expo[mode] = dict(dobs=float(np.mean(dobs_evt)), dobs_sd=float(np.std(dobs_evt)),
                      mag=float(np.mean(mags)), acc=len(dobs_evt) / (EXPO_N * (SIM4.n_frames - 1)))
print("=== E1 — affordance perceptual magnitude (clean obs; vs Exp-2 nudge |Δobs| 0.03–0.13) ===")
both_table(expo, [("mag", "mean move (world units)", "{:.2f}"), ("dobs", "mean |Δobs| per event", "{:.3f}"),
                  ("dobs_sd", "±sd", "{:.3f}"), ("acc", "accepted frac/transition", "{:.3f}")], row_hdr="affordance")

fig, ax = plt.subplots(figsize=(7.5, 4.2))
ms = ["dxdy", "teleport", "axis_x"]; xs = np.arange(len(ms))
ax.bar(xs, [expo[m]["dobs"] for m in ms], color=[MODE_COLOR[m] for m in ms],
       yerr=[expo[m]["dobs_sd"] for m in ms], capsize=4)
ax.axhspan(0.03, 0.13, color="0.7", alpha=0.35, label="Exp-2 nudge range (0.03–0.13)")
for x, m in zip(xs, ms): ax.text(x, expo[m]["dobs"] + 0.004, f"{expo[m]['dobs']:.3f}", ha="center", fontsize=9)
ax.set_xticks(xs); ax.set_xticklabels(ms); ax.set_ylabel("mean per-event |Δobs| (intensity)")
ax.set_title("Fig E1 — affordance perceptual magnitude (continuous actions are large)"); ax.legend(fontsize=8); style_ax(ax)
fig.tight_layout(); fig.savefig(f"{OUT}/figE1_magnitude.png", dpi=130, bbox_inches="tight"); display(fig); plt.close(fig)


In [ ]:
# [E2] Model-quality check — did each action model learn to reproduce the ACTED world? (replaces the old Δ-only plot)
#      GT (sim) vs the model's teacher-forced next-step prediction vs its free-run, per action model, with the true
#      actions fed in and action frames marked. Lets us verify the "no editability" conclusion isn't garbage-in.
from dataclasses import replace as _dcr
from pim.simulator.renderer import render_scene
T_MQ, W_MQ = 40, 6
def _render_noise(scene, noise):
    return render_scene(_dcr(scene, config=_dcr(scene.config, obs_noise_std=noise)))[2].astype(np.float32)
@torch.no_grad()
def _tf_pred(m, obs_noisy, acts):
    st = None; preds = [obs_noisy[0]]
    for t in range(T_MQ - 1):
        p, st = m.step(torch.from_numpy(obs_noisy[t])[None].float().to(DEVICE), st,
                       action=torch.from_numpy(acts[t])[None].float().to(DEVICE)); preds.append(p.squeeze(0).cpu().numpy())
    return np.stack(preds)
@torch.no_grad()
def _free_run(m, obs_noisy, acts):
    st = None
    for t in range(W_MQ):
        _, st = m.step(torch.from_numpy(obs_noisy[t])[None].float().to(DEVICE), st, action=torch.from_numpy(acts[t])[None].float().to(DEVICE))
    out = list(obs_noisy[:W_MQ]); cur = torch.from_numpy(obs_noisy[W_MQ - 1])[None].float().to(DEVICE)
    for t in range(W_MQ - 1, T_MQ - 1):
        p, st = m.step(cur, st, action=torch.from_numpy(acts[t])[None].float().to(DEVICE)); cur = p; out.append(p.squeeze(0).cpu().numpy())
    return np.stack(out)

DARKb = "#0a0a14"; e2 = {}
fig, axes = plt.subplots(len(ACTION_MODELS), 3, figsize=(11, 3.5 * len(ACTION_MODELS)), facecolor=DARKb, squeeze=False)
for r, (name, mode) in enumerate(ACTION_MODELS.items()):
    m = RAW[name][0]
    cfg = _dcr(SIM4, seed=40 + r, n_frames=T_MQ)
    scene, acts = simulate_with_continuous_actions(cfg, mode=mode, move_scale=4.0, p_action=0.30, action_seed=40 + r + 900)
    gt_clean = _render_noise(scene, 0.0); gt_noisy = _render_noise(scene, float(SIM4.obs_noise_std))
    tfp = _tf_pred(m, gt_noisy, acts); fr = _free_run(m, gt_noisy, acts)
    afr = [s for s in range(T_MQ) if acts[s, :, 0].any()]
    e2[name] = float(np.abs(tfp[1:] - gt_clean[1:]).mean())      # teacher-forced-pred fidelity to the true acted world
    for c, (img, ttl) in enumerate([(gt_clean, "GT (sim, clean)"), (tfp, "model teacher-forced pred"),
                                    (fr, f"model free-run (warm {W_MQ})")]):
        ax = axes[r][c]; ax.set_facecolor(DARKb)
        ax.imshow(np.clip(img, 0, 1), aspect="auto", origin="upper", cmap="gray", vmin=0, vmax=1, interpolation="nearest")
        for s in afr: ax.axhline(s + 0.5, color="#fa8850", lw=0.9, ls="--", alpha=0.7)
        if r == 0: ax.set_title(ttl, color="w", fontsize=10)
        if c == 0: ax.set_ylabel(f"{name} ({mode})\nframe", color="w", fontsize=9)
        ax.set_xlabel("ray", color="w", fontsize=8); ax.tick_params(colors="0.7", labelsize=7)
fig.legend(handles=[Line2D([0],[0], color="#fa8850", ls="--", lw=2, label="action frame (input to the model)")],
           loc="upper center", ncol=1, fontsize=9, frameon=False, labelcolor="w", bbox_to_anchor=(0.5, 0.985))
fig.suptitle("Fig E2 \u2014 did each action model learn to reproduce the ACTED world? GT vs teacher-forced pred vs free-run",
             color="w", y=1.0, fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.96]); fig.savefig(f"{OUT}/figE2_model_learned_task.png", dpi=130, bbox_inches="tight", facecolor=DARKb)
display(fig); plt.close(fig)
print("action-model fidelity (mean|teacher-forced pred \u2212 GT clean|, lower = learned the acted world better): "
      + " | ".join(f"{n}: {v:.3f}" for n, v in e2.items()))

In [ ]:
# [4] Passive (no-op) teacher-force ALL FIVE models on the dataset-4 test subset -> hidden-state banks (identical data).
MODEL_ORDER = ["Baseline", "Perturbed-passive (teleport)", "M_dxdy", "M_teleport", "M_axis"]
MCOLOR = {"Baseline": OK["grey"], "Perturbed-passive (teleport)": OK["blue"], "M_dxdy": OK["green"],
          "M_teleport": OK["orange"], "M_axis": OK["violet"]}
SHORT = {"Baseline": "Base", "Perturbed-passive (teleport)": "Pert-pass", "M_dxdy": "M_dxdy",
         "M_teleport": "M_tele", "M_axis": "M_axis"}
MODELS = {}
for name in MODEL_ORDER:
    m = RAW[name][0]
    preds, states = ev.teacher_force(m, sub_loader, device=DEVICE)   # passive: observe_sequence -> no-op action
    MODELS[name] = dict(model=m, states=states, color=MCOLOR[name])
    print(f"{name:28s} states {states.shape} | passive next-step MSE "
          f"{float(((preds - obs_eval[:, 1:, :]) ** 2).mean()):.5f}")


## §1 — Geometry of the passive state manifold (lighter structure context)

Intrinsic dimension (TwoNN, MLE), linear-hull dimension, and local curvature of the visited-`h` bank, in **passive
(no-op)** mode for each model. Physical reference = 8 DOF. Does any affordance reshape the passive manifold?

In [ ]:
# [5] §1 Geometry — PCA scree + model-free intrinsic dim (TwoNN, MLE) + tangent-rotation curvature, per model.
def scree(states):
    bank = torch.from_numpy(states.reshape(-1, H)).float().to(DEVICE)
    cum = np.cumsum(_pca_subspace(bank, n_components=H, var_threshold=1.0).explained_variance_ratio.cpu().numpy())
    return cum, {p: int((cum < p).sum()) + 1 for p in (0.70, 0.90, 0.95)}
@torch.no_grad()
def _knn(Q, X, k, chunk=2000):
    out = []
    for i in range(0, Q.shape[0], chunk):
        out.append(torch.topk(torch.cdist(Q[i:i + chunk], X), k + 1, largest=False, dim=1).values)
    return torch.cat(out, 0)
@torch.no_grad()
def two_nn_id(X, sample=10000, seed=0):
    g = torch.Generator(device=X.device).manual_seed(seed)
    idx = torch.randperm(X.shape[0], generator=g, device=X.device)[:min(sample, X.shape[0])]
    v = _knn(X[idx], X, 2); r1, r2 = v[:, 1], v[:, 2]; keep = (r1 > 1e-9) & (r2 > r1)
    return float(1.0 / torch.log(r2[keep] / r1[keep]).mean())
@torch.no_grad()
def mle_id(X, k=20, sample=10000, seed=0):
    g = torch.Generator(device=X.device).manual_seed(seed)
    idx = torch.randperm(X.shape[0], generator=g, device=X.device)[:min(sample, X.shape[0])]
    v = _knn(X[idx], X, k); logT = torch.log(v[:, 1:k + 1].clamp_min(1e-9))
    mk = 1.0 / ((logT[:, k - 1:k] - logT[:, :k - 1]).mean(1)).clamp_min(1e-9)
    return float(mk.mean() * (k - 2) / (k - 1))
@torch.no_grad()
def tangent_curv(X, n_anchors=150, k=64, n_tan=8, seed=0):
    g = torch.Generator(device=X.device).manual_seed(seed)
    aidx = torch.randperm(X.shape[0], generator=g, device=X.device)[:n_anchors]
    def tan(c):
        idx = torch.topk(torch.cdist(X[c:c + 1], X)[0], k + 1, largest=False).indices[1:]
        P = X[idx] - X[idx].mean(0, keepdim=True)
        return torch.linalg.svd(P, full_matrices=False)[2][:n_tan]
    angs = []
    for i in aidx:
        nn1 = int(torch.topk(torch.cdist(X[i:i + 1], X)[0], 2, largest=False).indices[1])
        s = torch.linalg.svdvals(tan(int(i)) @ tan(nn1).T).clamp(-1, 1)
        angs.append(float(torch.rad2deg(torch.arccos(s)).mean()))
    return float(np.mean(angs))
PHYS_DOF = N_OBJ * 4; GEO = {}
for name in MODEL_ORDER:
    st = MODELS[name]["states"]; bank = torch.from_numpy(st.reshape(-1, H)).float().to(DEVICE)
    cum, dims = scree(st)
    GEO[name] = dict(cum=cum, hull90=dims[0.90], hull95=dims[0.95],
                     twonn=two_nn_id(bank), mle=mle_id(bank), curv=tangent_curv(bank))
    del bank
    if DEVICE == "cuda": torch.cuda.empty_cache()
print("=== §1 GEOMETRY (passive no-op banks; physical DOF = 8) ===")
both_table({n: GEO[n] for n in MODEL_ORDER},
    [("hull90", "PCA hull @90%", "{:d}"), ("hull95", "PCA hull @95%", "{:d}"),
     ("twonn", "intrinsic TwoNN", "{:.2f}"), ("mle", "intrinsic MLE", "{:.2f}"),
     ("curv", "tangent rot (deg)", "{:.1f}")], row_hdr="model (passive)")


In [ ]:
# [6] Fig 1 — geometry (all five passive models side by side): (a) PCA scree, (b) intrinsic dim vs hull, (c) curvature.
fig, axes = plt.subplots(1, 3, figsize=(17, 4.4))
ax = axes[0]
for n in MODEL_ORDER:
    ax.plot(np.arange(1, len(GEO[n]["cum"]) + 1), GEO[n]["cum"], color=MCOLOR[n], lw=2, label=SHORT[n])
    ax.axvline(GEO[n]["hull90"], color=MCOLOR[n], ls="--", lw=1)
ax.axhline(0.90, color="0.6", ls=":", lw=1); ax.set_xlim(0, 80); ax.set_xlabel("# PCA components")
ax.set_ylabel("cumulative variance"); ax.set_title("(a) PCA scree (dashed = hull@90%)"); ax.legend(fontsize=8); style_ax(ax)
ax = axes[1]
cats = ["TwoNN", "MLE", "hull@90%"]; x = np.arange(3); w = 0.8 / len(MODEL_ORDER)
for j, n in enumerate(MODEL_ORDER):
    vals = [GEO[n]["twonn"], GEO[n]["mle"], GEO[n]["hull90"]]; off = (j - (len(MODEL_ORDER) - 1) / 2) * w
    ax.bar(x + off, vals, w, color=MCOLOR[n], label=SHORT[n])
ax.axhline(PHYS_DOF, color="k", ls="--", lw=1.4, label=f"physical {PHYS_DOF} DOF")
ax.set_xticks(x); ax.set_xticklabels(cats); ax.set_ylabel("dimension")
ax.set_title("(b) intrinsic dim vs linear hull@90%"); ax.legend(fontsize=7); style_ax(ax)
ax = axes[2]
cv = [GEO[n]["curv"] for n in MODEL_ORDER]
ax.bar(range(len(MODEL_ORDER)), cv, color=[MCOLOR[n] for n in MODEL_ORDER])
for xi, v in enumerate(cv): ax.text(xi, v + 0.5, f"{v:.0f}°", ha="center", fontsize=8)
ax.set_xticks(range(len(MODEL_ORDER))); ax.set_xticklabels([SHORT[n] for n in MODEL_ORDER], fontsize=8, rotation=15)
ax.set_ylabel("tangent rotation @NN (deg)"); ax.set_title("(c) local curvature"); style_ax(ax)
fig.suptitle("Fig 1 — Passive state-manifold geometry (five models)", y=1.03, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig1_geometry.png", dpi=130, bbox_inches="tight"); display(fig); plt.close(fig)


## §2 — Recoverability of (pos, vel) from the passive hidden state

Linear and MLP probes read the physical state out of a single passive `h`, split early-t (t<15) / late-t (t≥15).
More individuated objects ⇒ *more* recoverable passive readout (higher R²), especially for velocity.

In [ ]:
# [7] §2 Recoverability — linear + MLP probes for (pos, vel); velocity early-t vs late-t; per model.
VCOMP = ["vx0", "vy0", "vx1", "vy1"]
def build_feats(states, regime):
    sf = states[:, 1:, :]
    y = velflat_tf[:, 1:, :]; mask = vis_tf[:, 1:] & vis_tf[:, :-1]; rm = np.zeros_like(mask)
    if regime == "early": rm[:, :LATE_T - 1] = True
    else: rm[:, LATE_T - 1:] = True
    return sf, y, (mask & rm)
def run_vel(states, regime):
    sf, y, mask = build_feats(states, regime)
    return {"lin": fit_probe(sf, y, mask, "linear"), "mlp": fit_probe(sf, y, mask, "mlp")}
vel_res, pos_res = {}, {}
for name in MODEL_ORDER:
    st = MODELS[name]["states"]
    for reg in ["early", "late"]: vel_res[(name, reg)] = run_vel(st, reg)
    pos_res[(name, "lin")] = fit_probe(st, posflat_tf, vis_tf, "linear")
    pos_res[(name, "mlp")] = fit_probe(st, posflat_tf, vis_tf, "mlp")
print("=== §2 RECOVERABILITY R² (in-sample; higher is better) ===")
rec = {}
for name in MODEL_ORDER:
    o = vel_res[(name, "late")]; oe = vel_res[(name, "early")]
    rec[name] = dict(pos_lin=pos_res[(name, "lin")]["r2"], pos_mlp=pos_res[(name, "mlp")]["r2"],
                     vel_lin=o["lin"]["r2"], vel_mlp=o["mlp"]["r2"], vel_mlp_e=oe["mlp"]["r2"])
both_table(rec, [("pos_lin", "pos R² lin", "{:.3f}"), ("pos_mlp", "pos R² MLP", "{:.3f}"),
    ("vel_lin", "vel R² lin (late)", "{:.3f}"), ("vel_mlp", "vel R² MLP (late)", "{:.3f}"),
    ("vel_mlp_e", "vel R² MLP (early)", "{:.3f}")], row_hdr="model (passive)")


In [ ]:
# [8] Fig 2 — recoverability: (a) position lin vs MLP, (b) velocity MLP early vs late, (c) per-component velocity (late MLP).
fig, axes = plt.subplots(1, 3, figsize=(17, 4.4)); x = np.arange(len(MODEL_ORDER)); w = 0.38
ax = axes[0]
ax.bar(x - w / 2, [pos_res[(n, "lin")]["r2"] for n in MODEL_ORDER], w, color=OK["sky"], label="linear")
ax.bar(x + w / 2, [pos_res[(n, "mlp")]["r2"] for n in MODEL_ORDER], w, color=OK["orange"], label="MLP")
ax.set_xticks(x); ax.set_xticklabels([SHORT[n] for n in MODEL_ORDER], fontsize=8, rotation=15); ax.set_ylim(0, 1.02)
ax.set_ylabel("position R²"); ax.set_title("(a) position readout"); ax.legend(fontsize=8); style_ax(ax)
ax = axes[1]
ax.bar(x - w / 2, [vel_res[(n, "early")]["mlp"]["r2"] for n in MODEL_ORDER], w, color="0.6", label="early-t")
ax.bar(x + w / 2, [vel_res[(n, "late")]["mlp"]["r2"] for n in MODEL_ORDER], w, color=OK["green"], label="late-t")
ax.set_xticks(x); ax.set_xticklabels([SHORT[n] for n in MODEL_ORDER], fontsize=8, rotation=15); ax.set_ylim(0, 1.02)
ax.set_ylabel("velocity R² (single-frame MLP)"); ax.set_title("(b) velocity readout"); ax.legend(fontsize=8); style_ax(ax)
ax = axes[2]; xc = np.arange(4); w2 = 0.8 / len(MODEL_ORDER)
for j, n in enumerate(MODEL_ORDER):
    ax.bar(xc + (j - (len(MODEL_ORDER) - 1) / 2) * w2, vel_res[(n, "late")]["mlp"]["r2pc"], w2, color=MCOLOR[n], label=SHORT[n])
ax.set_xticks(xc); ax.set_xticklabels(VCOMP); ax.set_ylim(0, 1.02)
ax.set_ylabel("velocity R² per component"); ax.set_title("(c) per-component velocity (late MLP)"); ax.legend(fontsize=7); style_ax(ax)
fig.suptitle("Fig 2 — Recoverability of (pos, vel) from a single passive hidden state", y=1.03, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig2_recoverability.png", dpi=130, bbox_inches="tight"); display(fig); plt.close(fig)


## §3 — Fiber collapse / canonicality

Is the passive hidden state a function of the 8-dim physical `(pos, vel)`? Fit `g(pos,vel)→h` and report the residual
fraction `‖h − g‖/‖h‖`. Lower = closer to a canonical function of the physical cause — structure editing needs.

In [ ]:
# [9] §3 Fiber collapse — residual of the best g(pos,vel)->h (linear & MLP); lower = more canonical.
def fit_g(block, kind):
    _, r2, _, rfrac = _fit_regress(posvel_tf[vis_tf], block[vis_tf], kind, hidden=512, n_epochs=100, lr=1.5e-3)
    return rfrac, r2
fiber = {}
for name in MODEL_ORDER:
    st = MODELS[name]["states"]; lrf, lr2 = fit_g(st, "linear"); mrf, mr2 = fit_g(st, "mlp")
    fiber[name] = dict(lin_rf=lrf, mlp_rf=mrf, mlp_r2=mr2, lin_drop=lrf - mrf)
print("=== §3 FIBER RESIDUAL ||h - g(pos,vel)|| / ||h||  (0 = fully canonical / a function of the 8-dim state) ===")
both_table({n: fiber[n] for n in MODEL_ORDER},
    [("lin_rf", "linear g resid", "{:.3f}"), ("mlp_rf", "MLP g resid", "{:.3f}"),
     ("lin_drop", "linear->MLP drop", "{:+.3f}"), ("mlp_r2", "MLP g R² on h", "{:.3f}")], row_hdr="model (passive)")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2)); x = np.arange(len(MODEL_ORDER)); w = 0.38
ax = axes[0]
ax.bar(x - w / 2, [fiber[n]["lin_rf"] for n in MODEL_ORDER], w, color=OK["sky"], label="linear g")
ax.bar(x + w / 2, [fiber[n]["mlp_rf"] for n in MODEL_ORDER], w, color=OK["orange"], label="MLP g")
ax.set_xticks(x); ax.set_xticklabels([SHORT[n] for n in MODEL_ORDER], fontsize=8, rotation=15)
ax.set_ylabel("residual fraction ||h - g|| / ||h||"); ax.set_ylim(0, 1.05)
ax.set_title("(a) is h a function of (pos, vel)?  lower = more canonical"); ax.legend(fontsize=8); style_ax(ax)
ax = axes[1]
ax.bar(x, [fiber[n]["mlp_r2"] for n in MODEL_ORDER], color=[MCOLOR[n] for n in MODEL_ORDER])
for xi, n in zip(x, MODEL_ORDER): ax.text(xi, fiber[n]["mlp_r2"] + 0.01, f"{fiber[n]['mlp_r2']:.2f}", ha="center", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels([SHORT[n] for n in MODEL_ORDER], fontsize=8, rotation=15); ax.set_ylim(0, 1.0)
ax.set_ylabel("MLP g: R² on h"); ax.set_title("(b) variance of h explained by g(pos, vel)"); style_ax(ax)
fig.suptitle("Fig 3 — Fiber collapse / canonicality of the passive hidden state", y=1.03, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig3_fiber.png", dpi=130, bbox_inches="tight"); display(fig); plt.close(fig)


## §4 — Object-handle selectivity + interface generalization (the headline)

On each **passive** model, warm up to the dataset-4 teleport edit frame, then apply the master editor line-up targeting
**object k's** position (the edit object), roll out, and score the handle in **observation space**: **reach** (does
object k's rays move toward its target), **collateral** (are the OTHER object's rays left alone), **ghost** (does k's
old location clear), **persistence** (does it hold over K=15). The editors are a **different write-mechanism** than the
trained action channel, so a positive result = the affordance moved object-hood **into the state** (interface
generalization), not a memorised button. Confound triad: Baseline → Perturbed-passive → M_teleport.

In [ ]:
# [10] §4 shared edit-set setup (model-independent): teleport targets, static renders, ghost/target/COLLATERAL rays, GT traj.
from pim.simulator.sim import Scene
from pim.simulator.renderer import render_scene
SUBSPACE_VAR, LOCAL_VAR, LOCAL_BANK_SIZE = 0.90, 0.90, 40_000
LOCAL_K_GEO = 64; N_EDIT, N_ROLLOUT = 48, 15; K_GEO_ITERS = 60; N_CTX = 6
N = min(N_EDIT, edits.n_samples); ef = edits.edit_frame
tgt_pos_flat = edits.positions[:N, ef, :N_OBJ, :].reshape(N, N_OBJ * 2).astype(np.float32)
vel_edits = h5py.File(edits.h5_path, "r")["velocities"][:N, ef, :N_OBJ, :].astype(np.float32)
tgt = torch.from_numpy(tgt_pos_flat).float().to(DEVICE)
tgt_pv = torch.from_numpy(np.concatenate([tgt_pos_flat, vel_edits.reshape(N, N_OBJ * 2)], 1)).float().to(DEVICE)
gt_traj_obs = edits.clean_obs[:N, ef:ef + N_ROLLOUT, :].astype(np.float32)     # true post-edit clean obs (time-evolving GT)
ctx_obs = edits.obs[:N, ef - N_CTX:ef, :].astype(np.float32)                   # shared pre-edit context (noisy, teacher-forced)
gt_obs_edit_frame = torch.from_numpy(edits.clean_obs[:N, ef, :]).float().to(DEVICE)
OBS_RES = gt_traj_obs.shape[-1]
sim = test.config["dataset"]["sim"]
cfg1 = SimConfig(seed=0, y_near=sim["y_near"], y_far=sim["y_far"], x_near=sim["x_near"], x_far=sim["x_far"],
                 n_objects=N_OBJ, radius=sim["radius"], n_frames=1, dt=sim["dt"], obs_res=sim["obs_res"],
                 refl_min=sim["refl_min"], refl_max=sim["refl_max"], fixed_reflectivities=True,
                 obs_noise_std=0.0, boundary="open", always_in_frustum=False)
REFL = np.array([sim["refl_min"], sim["refl_max"]], np.float32); RAD = np.array([sim["radius"]] * N_OBJ, np.float32)
COLc = np.tile(np.array([[1, 1, 1]], np.float32), (N_OBJ, 1))
tgt_pos = edits.positions[:N, ef, :N_OBJ, :].astype(np.float32); pre_pos = edits.positions[:N, ef - 1, :N_OBJ, :].astype(np.float32)
tgt_render_id = np.zeros((N, OBS_RES), np.int64); tgt_render_int = np.zeros((N, OBS_RES), np.float32); pre_render_id = np.zeros((N, OBS_RES), np.int64)
for i in range(N):
    sc = Scene(positions=tgt_pos[i][None], velocities=np.zeros((1, N_OBJ, 2), np.float32), radii=RAD, colors=COLc, reflectivities=REFL, config=cfg1)
    _, rid, rint = render_scene(sc); tgt_render_id[i], tgt_render_int[i] = rid[0], rint[0]
    scp = Scene(positions=pre_pos[i][None], velocities=np.zeros((1, N_OBJ, 2), np.float32), radii=RAD, colors=COLc, reflectivities=REFL, config=cfg1)
    _, ridp, _ = render_scene(scp); pre_render_id[i] = ridp[0]
edit_obj = edits.edit_object[:N].astype(int); other_obj = (1 - edit_obj).astype(int)   # 2-object world
ghost_mask = np.zeros((N, OBS_RES), bool); k_mask = np.zeros((N, OBS_RES), bool); collat_mask = np.zeros((N, OBS_RES), bool)
for i in range(N):
    ghost_mask[i]  = (pre_render_id[i] == edit_obj[i]) & (tgt_render_id[i] != edit_obj[i])   # vacated rays
    k_mask[i]      = (tgt_render_id[i] == edit_obj[i]) | ghost_mask[i]                        # object-k rays (target ∪ vacated)
    collat_mask[i] = (tgt_render_id[i] == other_obj[i])                                       # the OTHER object's rays
def centroid(mask_row):
    idx = np.where(mask_row)[0]; return idx.mean() if idx.size else np.nan
teleport = np.linalg.norm(tgt_pos - pre_pos, axis=-1)[np.arange(N), edit_obj]
has_ghost = ghost_mask.sum(1) >= 3; SAMPLES = list(np.argsort(teleport * has_ghost)[::-1][:2])
print(f"N={N} edit_frame={ef} rollout={N_ROLLOUT} | ghost rays {int(ghost_mask.sum())} | "
      f"k rays {int(k_mask.sum())} | collateral rays {int(collat_mask.sum())} | waterfall samples {SAMPLES}")


In [ ]:
# [11] §4 per-model prep: warm-up h0, linear pos probe, frozen MLP (pos,vel) probe, global-PCA subspace, local bank, true-state-swap.
@torch.no_grad()
def tf_hidden_at(model, obs_seqs, frame):
    out = np.zeros((obs_seqs.shape[0], H), np.float32)
    for i in range(obs_seqs.shape[0]):
        ot = torch.from_numpy(obs_seqs[i]).float().to(DEVICE); state = None
        for t in range(frame + 1):
            _, state = model.step(ot[t].unsqueeze(0), state)      # passive no-op
        out[i] = model.flat_state(state).squeeze(0).cpu().numpy()
    return out
def prep_model(model, states, name):
    sdef = StateDefinition(name="positions", state_shape=(N_OBJ, 2), extract_fn=lambda b: b["positions"])
    lin = LinearExtractor(H, sdef, use_lstsq=True); lin.fit(states, pos_tf, mask=vis_tf, device=DEVICE); lin = lin.to(DEVICE).eval()
    A, b_, A_pinv = probe_decomposition(lin)
    sdef_pv = StateDefinition(name="posvel", state_shape=(N_OBJ * 4,), extract_fn=lambda b: b)
    mlp_pv = MLPExtractor(H, sdef_pv, mlp_hidden=128, n_epochs=30, lr=5e-3)
    mlp_pv.fit(states, posvel_tf, mask=vis_tf, device=DEVICE); mlp_pv = mlp_pv.to(DEVICE).eval()
    sub = fit_state_subspace(states, var_threshold=SUBSPACE_VAR)
    sub = replace(sub, mean=sub.mean.to(DEVICE), basis=sub.basis.to(DEVICE),
                  explained_variance_ratio=sub.explained_variance_ratio.to(DEVICE))
    ba = states.reshape(-1, H)
    bidx = np.random.RandomState(0).choice(ba.shape[0], size=min(LOCAL_BANK_SIZE, ba.shape[0]), replace=False)
    bank = torch.from_numpy(ba[bidx]).float().to(DEVICE)
    warm = ev.warm_up_to_edit(model, edits.obs[:N], ef, n_viz=N, n_ctx_show=8, device=DEVICE)
    h0 = torch.from_numpy(warm.h_at_edit[:N]).float().to(DEVICE)
    h_swap = torch.from_numpy(tf_hidden_at(model, edits.obs[:N], ef)).float().to(DEVICE)
    return dict(model=model, A=A, b=b_, A_pinv=A_pinv, mlp_pv=mlp_pv, sub=sub, bank=bank, h0=h0, h_swap=h_swap)
WM = {name: prep_model(MODELS[name]["model"], MODELS[name]["states"], name) for name in MODEL_ORDER}
def readout(P, h): return h @ P["A"].T + P["b"]
def readout_rmse(P, h): return float((readout(P, h) - tgt).pow(2).mean().sqrt())
for n in MODEL_ORDER:
    print(f"{n:28s} un-edited readout RMSE {readout_rmse(WM[n], WM[n]['h0']):.3f} | "
          f"true-state-swap readout RMSE {readout_rmse(WM[n], WM[n]['h_swap']):.3f}")


In [ ]:
# [12] §4 real-state references: leave-out local-PCA residual (excludes the query's own NN) + global-PCA hull residual.
@torch.no_grad()
def loo_local_resid(h_batch, bank, k_neighbors=LOCAL_K_GEO, n_probe=100, var_threshold=LOCAL_VAR):
    hb = h_batch if isinstance(h_batch, torch.Tensor) else torch.as_tensor(h_batch, device=DEVICE, dtype=torch.float32)
    fracs = []
    for i in range(min(n_probe, hb.shape[0])):
        q = hb[i].reshape(-1); d = torch.cdist(q[None], bank)[0]
        idx = torch.topk(d, min(k_neighbors + 1, bank.shape[0]), largest=False).indices[1:]
        sub = _pca_subspace(bank[idx], n_components=None, var_threshold=var_threshold)
        proj = project_to_subspace(q[None], sub)[0]
        fracs.append(float((q - proj).norm()) / max(float((q - sub.mean).norm()), 1e-9))
    return float(np.mean(fracs))
REAL_LOO, REAL_GLOB = {}, {}
for n in MODEL_ORDER:
    P = WM[n]
    REAL_LOO[n] = loo_local_resid(P["bank"][:200], P["bank"], n_probe=200)
    REAL_GLOB[n] = float(offmanifold_residual(P["bank"][:2000], P["sub"]).mean())
    print(f"{n:28s} real-state leave-out local-PCA resid {REAL_LOO[n]:.3f} | global-PCA hull resid {REAL_GLOB[n]:.3f}")


In [ ]:
# [13] §4 run the five master editors on all five passive models.
ED_ORDER = ["Readout injection", "MLP-probe gradient", "Global-PCA projection", "PCA geodesic", "Decoder gradient"]
@torch.no_grad()
def pca_geodesic(P, h_start, target, k_local=LOCAL_K_GEO, const_step=None, k_iters=K_GEO_ITERS, desc="geo"):
    A, b_, A_pinv, bank = P["A"], P["b"], P["A_pinv"], P["bank"]
    if const_step is None:
        const_step = 0.34 * float((inject_state(h_start, target, A, A_pinv, b_) - h_start).norm(dim=-1).mean())
    Nn = h_start.shape[0]; h_out = torch.empty_like(h_start)
    for i in range(Nn):
        h = h_start[i:i + 1]; t = target[i:i + 1]
        for kk in range(k_iters):
            d = inject_state(h, t, A, A_pinv, b_) - h; nrm = d.norm(); dhat = d / nrm if float(nrm) > 1e-12 else d
            h_step = h + const_step * dhat
            sub = fit_local_subspace(bank, h_step[0], k_neighbors=k_local, var_threshold=LOCAL_VAR, bank_size=LOCAL_BANK_SIZE)
            h = project_to_subspace(h_step, sub)
        h_out[i] = h[0]
    return h_out
def decoder_grad_edit(model, h_init, target_obs, n_iter=250, lr=0.05):
    h = h_init.clone().detach().requires_grad_(True); opt = torch.optim.Adam([h], lr=lr)
    with torch.backends.cudnn.flags(enabled=False):
        for _ in range(n_iter):
            loss = ((model.decode(model.state_from_flat(h)) - target_obs) ** 2).mean()
            opt.zero_grad(); loss.backward(); opt.step()
    return h.detach(), float(loss.item())
EDITS4 = {}
for n in tqdm(MODEL_ORDER, desc="editors x models"):
    P = WM[n]; h0 = P["h0"]; A, b_, A_pinv = P["A"], P["b"], P["A_pinv"]; E = {}
    E["Readout injection"] = inject_state(h0, tgt, A, A_pinv, b_)
    outs = []
    for i in range(N):
        h_i, _ = gradient_steer(h0[i:i + 1], tgt_pv[i:i + 1], P["mlp_pv"], n_steps=150, lr=0.01); outs.append(h_i)
    E["MLP-probe gradient"] = torch.cat(outs, 0)
    E["Global-PCA projection"] = manifold_steer(h0, tgt, lambda h, t: inject_state(h, t, A, A_pinv, b_), P["sub"], n_iters=50)
    E["PCA geodesic"] = pca_geodesic(P, h0, tgt, desc=f"geodesic {n}")
    E["Decoder gradient"], dl = decoder_grad_edit(P["model"], h0, gt_obs_edit_frame)
    EDITS4[n] = E
    print(f"{n:28s} dec MSE {dl:.5f} | readout RMSE: " + " ".join(f"{k.split()[0]}={readout_rmse(P, h):.2f}" for k, h in E.items()))


In [ ]:
# [14] §4 model rollouts from every reference/editor state (passive free-run; step-0 = decode without advancing).
REF_ORDER = ["Unsteered", "True-state swap"]
COL = {"GT (sim)": "k", "Unsteered": OK["grey"], "True-state swap": OK["sky"],
       "Readout injection": OK["yellow"], "MLP-probe gradient": OK["orange"],
       "Global-PCA projection": OK["green"], "PCA geodesic": OK["blue"], "Decoder gradient": OK["pink"]}
@torch.no_grad()
def rollout_from_flat(model, h_array, n_rollout):
    out = []
    for i in range(h_array.shape[0]):
        h = torch.as_tensor(h_array[i], dtype=torch.float32, device=DEVICE).unsqueeze(0)
        o, _ = _rollout(model, h, n_rollout); out.append(o)
    return np.stack(out)
ROLL = {}
for n in MODEL_ORDER:
    P = WM[n]; states4 = {"Unsteered": P["h0"], "True-state swap": P["h_swap"], **EDITS4[n]}
    ROLL[n] = {k: rollout_from_flat(P["model"], h.detach().cpu().numpy(), N_ROLLOUT) for k, h in states4.items()}
print("rollouts done:", {SHORT[n]: ROLL[n]["Unsteered"].shape for n in MODEL_ORDER})


In [ ]:
# [15] §4 object-handle metric suite (reach / collateral / ghost / persistence + readout / next-step / manifold), per model.
def rms(a, b): return float(np.sqrt(((a - b) ** 2).mean()))
def rms_mask(a, b, mask):
    # a, b: (N, R); mask: (N, R) bool (per-sample ray set). RMSE over all masked (sample, ray) cells.
    if mask.sum() == 0: return float("nan")
    d = (a - b) ** 2
    return float(np.sqrt(d[mask].mean()))
METRICS, STEP_RMSE, SCARD = {}, {}, {}
for n in MODEL_ORDER:
    P = WM[n]; obs_u = ROLL[n]["Unsteered"]; swap = ROLL[n]["True-state swap"]
    ref_reach = rms_mask(swap[:, 0, :], obs_u[:, 0, :], k_mask)      # what a real swap does to obj-k rays
    rows = {}
    for k in REF_ORDER + ED_ORDER:
        o = ROLL[n][k]
        h = {"Unsteered": P["h0"], "True-state swap": P["h_swap"]}.get(k)
        h = EDITS4[n][k] if h is None else h
        reach = 100 * rms_mask(o[:, 0, :], obs_u[:, 0, :], k_mask) / max(ref_reach, 1e-9)
        collat = 100 * rms_mask(o[:, 0, :], obs_u[:, 0, :], collat_mask) / max(ref_reach, 1e-9)
        ghost = (float(o[:, 0, :][ghost_mask].mean() / max(obs_u[:, 0, :][ghost_mask].mean(), 1e-6))
                 if ghost_mask.sum() > 0 else float("nan"))
        chg0 = rms(o[:, 0, :], obs_u[:, 0, :])
        chg_late = np.mean([rms(o[:, s, :], obs_u[:, s, :]) for s in range(10, N_ROLLOUT)])
        persist = chg_late / max(chg0, 1e-9)                 # anti-reversion: stays away from unsteered (>=1) or reverts (<1)
        gt_rmse = float(np.mean([rms(o[:, s, :], gt_traj_obs[:, s, :]) for s in range(N_ROLLOUT)]))  # fidelity to TRUE post-edit world
        rows[k] = dict(readout=readout_rmse(P, h), nextstep=rms(o[:, 1, :], gt_traj_obs[:, 1, :]),
                       reach=reach, collat=collat, ghost=ghost, persist=persist, gt_rmse=gt_rmse,
                       select=reach / max(reach + collat, 1e-9),
                       loo=loo_local_resid(h, P["bank"], n_probe=min(48, N)))
    METRICS[n] = rows
    STEP_RMSE[n] = {k: [rms(ROLL[n][k][:, s, :], gt_traj_obs[:, s, :]) for s in range(N_ROLLOUT)] for k in REF_ORDER + ED_ORDER}
# Scorecard uses ONE canonical structural editor for every model (apples-to-apples across the confound triad):
# PCA geodesic — the manifold-respecting structural editor (best for 4/5 models; a fixed choice, not per-model-best).
STRUCT_ED = ["Readout injection", "MLP-probe gradient", "Global-PCA projection", "PCA geodesic"]
SCORE_EDITOR = "PCA geodesic"
for n in MODEL_ORDER: SCARD[n] = METRICS[n][SCORE_EDITOR] | {"editor": SCORE_EDITOR}
M4 = [("reach", "reach %swap ↑", "{:.1f}"), ("collat", "collateral %swap ↓", "{:.1f}"),
      ("select", "selectivity ↑", "{:.2f}"), ("ghost", "ghost ratio ↓", "{:.3f}"),
      ("persist", "anti-reversion ↑", "{:.2f}"), ("gt_rmse", "GT-traj RMSE ↓", "{:.3f}"),
      ("readout", "readout RMSE ↓", "{:.3f}"),
      ("nextstep", "GT next-step RMSE ↓", "{:.3f}")]
for n in MODEL_ORDER:
    print(f"=== {n} — object-handle metrics (obj-k swap ref, all editors) ===")
    both_table(METRICS[n], M4, row_hdr=str(n))


In [ ]:
# [16] Fig 4 — OBJECT-HANDLE SELECTIVITY SCORECARD: best structural editor per model, all five models side by side.
fig, axes = plt.subplots(1, 4, figsize=(19, 4.6))
panels = [("reach", "reach (% of swap) ↑", 1), ("collat", "collateral (% of swap) ↓", 1),
          ("ghost", "ghost ratio ↓", 1), ("persist", "persistence ↑", 1)]
xs = np.arange(len(MODEL_ORDER))
for ax, (key, title, _) in zip(axes, panels):
    vals = [SCARD[n][key] for n in MODEL_ORDER]
    ax.bar(xs, vals, color=[MCOLOR[n] for n in MODEL_ORDER])
    for x, v in zip(xs, vals): ax.text(x, v + 0.01 * max(vals + [1]), (f"{v:.1f}" if key in ("reach","collat") else f"{v:.2f}"), ha="center", fontsize=8)
    if key == "reach": ax.axhline(100, color="0.3", ls="--", lw=1.1); ax.text(len(xs)-0.5, 101, "true swap 100%", ha="right", fontsize=7, color="0.3")
    if key == "ghost": ax.axhline(1.0, color="0.3", ls="--", lw=1.1); ax.text(len(xs)-0.5, 1.01, "unsteered 1.0", ha="right", fontsize=7, color="0.3")
    ax.set_xticks(xs); ax.set_xticklabels([SHORT[n] for n in MODEL_ORDER], fontsize=8, rotation=20)
    ax.set_title(title, fontsize=10); style_ax(ax)
fig.suptitle(f"Fig 4 — Object-handle scorecard on the PASSIVE latent ({SCORE_EDITOR}, same editor for every model; §4 editors = untrained write-mechanism)",
             y=1.03, fontsize=12)
fig.text(0.5, -0.02, "clean handle = high reach + LOW collateral + ghost≪1 + persistence≥1; a real teleport (true-state swap) = 100% reach",
         ha="center", fontsize=8, color="0.35")
fig.tight_layout(); fig.savefig(f"{OUT}/fig4_scorecard.png", dpi=130, bbox_inches="tight"); display(fig); plt.close(fig)


In [ ]:
# [17] Fig 4b — per-step GT-trajectory RMSE (does the edit track the time-evolving true post-edit obs?), one panel per model.
fig, axes = plt.subplots(1, len(MODEL_ORDER), figsize=(4.0 * len(MODEL_ORDER), 4.0), sharey=True)
steps = np.arange(N_ROLLOUT)
for ax, n in zip(axes, MODEL_ORDER):
    for k in REF_ORDER + ED_ORDER:
        ax.plot(steps, STEP_RMSE[n][k], color=COL[k], lw=1.5, marker="o", ms=2.5, label=k)
    ax.set_title(SHORT[n], fontsize=10, color=MCOLOR[n]); ax.set_xlabel("rollout step (0 = edit frame)"); style_ax(ax)
axes[0].set_ylabel("RMSE(generated, true post-edit obs)")
handles = [Line2D([0], [0], color=COL[k], lw=2, marker="o", ms=4, label=k) for k in REF_ORDER + ED_ORDER]
fig.legend(handles=handles, loc="upper center", ncol=7, fontsize=8.5, frameon=False, bbox_to_anchor=(0.5, 1.02))
fig.suptitle("Fig 4b — Per-step tracking of the time-evolving GT trajectory (passive rollout from each edited state)", y=1.09, fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.98]); fig.savefig(f"{OUT}/fig4b_perstep.png", dpi=130, bbox_inches="tight"); display(fig); plt.close(fig)


In [ ]:
# [18] Fig 5 — editor waterfalls (dark), one per model. Context (noisy) + the TEACHER-FORCED true edit frame (ef,
#      observed) + the free-run rollout (ef+1 onward). Showing ef teacher-forced fixes the GRU +1 decode offset so
#      every column aligns to sim frames honestly (the ef row is what was observed; each column's model rollout is below).
DARK_BG, DARK_TEXT, DARK_TICK, EDIT_LINE, TF_LINE = "#0a0a14", "#a3adc2", "#808a9d", "#fa8850", "#FFD166"
WATERFALL_COLS = ["GT (sim)"] + REF_ORDER + ED_ORDER
tf_ef = edits.clean_obs[:N, ef, :].astype(np.float32)          # true post-edit obs AT ef (teacher-forced / observed)
def editor_waterfall_fig(n, tag, fname):
    nc = len(WATERFALL_COLS)
    fig, axes = plt.subplots(len(SAMPLES), nc, figsize=(2.6 * nc, 3.3 * len(SAMPLES)), squeeze=False, facecolor=DARK_BG)
    for r, smp in enumerate(SAMPLES):
        tcx = centroid(tgt_render_id[smp] == edit_obj[smp]); pcx = centroid(pre_render_id[smp] == edit_obj[smp])
        ocx = centroid(tgt_render_id[smp] == other_obj[smp])
        for c, k in enumerate(WATERFALL_COLS):
            ax = axes[r][c]
            if k == "GT (sim)":
                ctx = edits.clean_obs[smp, ef - N_CTX:ef, :].astype(np.float32)
                roll = edits.clean_obs[smp, ef + 1:ef + N_ROLLOUT, :].astype(np.float32)        # ef+1 .. ef+N_ROLLOUT-1 (free-run ref)
            else:
                ctx = ctx_obs[smp]
                roll = ROLL[n][k][smp][1:]                                                       # free-run ef+1 onward (drop step0=ef; ef is the shared true row)
            panel = np.clip(np.concatenate([ctx, tf_ef[smp][None], roll], 0), 0, 1)             # ctx | ef (observed) | rollout
            ax.set_facecolor(DARK_BG)
            for sp in ax.spines.values(): sp.set_edgecolor(DARK_TICK)
            ax.imshow(panel, aspect="auto", origin="upper", cmap="gray", vmin=0, vmax=1, interpolation="nearest")
            ax.axhline(N_CTX - 0.5, color=EDIT_LINE, lw=1.2, ls="--", alpha=0.85)                # context -> edit frame
            ax.axhline(N_CTX + 0.5, color=TF_LINE, lw=1.1, ls=":", alpha=0.9)                    # edit frame -> free-run (ef+1)
            if not np.isnan(tcx): ax.axvline(tcx, color="#00E676", lw=1.4, alpha=0.9)
            if not np.isnan(pcx): ax.axvline(pcx, color="#FF5252", ls="--", lw=1.4, alpha=0.9)
            if not np.isnan(ocx): ax.axvline(ocx, color="#40C4FF", ls=":", lw=1.4, alpha=0.9)
            if r == 0: ax.set_title(k, fontsize=8.5, color=DARK_TEXT)
            if c == 0:
                ax.set_ylabel(f"sample {smp} (tel {teleport[smp]:.1f})\nsim frame", fontsize=8, color=DARK_TEXT)
                ax.set_yticks([0, N_CTX, N_CTX + 5, N_CTX + 10]); ax.set_yticklabels([ef - N_CTX, ef, ef + 5, ef + 10])
            else: ax.set_yticks([])
            ax.set_xlabel("ray", fontsize=8, color=DARK_TEXT); ax.tick_params(colors=DARK_TICK, labelsize=7)
    handles = [Line2D([0], [0], color="#00E676", lw=2.2, label="object-k target"),
               Line2D([0], [0], color="#FF5252", ls="--", lw=2.2, label="object-k ghost (pre-edit)"),
               Line2D([0], [0], color="#40C4FF", ls=":", lw=2.2, label="OTHER object (collateral)"),
               Line2D([0], [0], color=EDIT_LINE, ls="--", lw=2.2, label="edit frame"),
               Line2D([0], [0], color=TF_LINE, ls=":", lw=2.2, label="ef = true post-edit obs (edit target); rows below = free-run from the edited state (ef+1 →)")]
    fig.legend(handles=handles, loc="upper center", ncol=3, fontsize=8.5, frameon=False, labelcolor=DARK_TEXT, bbox_to_anchor=(0.5, 0.995))
    fig.suptitle(f"Fig {tag} — {n} passive-latent editor waterfalls (ef teacher-forced; rollout = free-run from ef+1)", y=1.0, fontsize=11, color=DARK_TEXT)
    fig.tight_layout(rect=[0, 0, 1, 0.94]); fig.savefig(f"{OUT}/{fname}", dpi=125, bbox_inches="tight", facecolor=DARK_BG); display(fig); plt.close(fig)
for name, tag in zip(MODEL_ORDER, ["5a", "5b", "5c", "5d", "5e"]):
    editor_waterfall_fig(name, tag, f"fig{tag}_waterfall_{SHORT[name].replace('-','').replace(' ','')}.png")

## Content generalization — `M_axis` (trained x-only) edited along **y** it never saw

The strongest individuation test: if the handle is a *real object* rather than a memorised x-affordance, then editing
the passive latent along **y** (never trained for `M_axis`) should work about as well as along **x** (trained). We
construct, from each edit sample's pre-edit position of the edit object, a pure **x-target** and a pure **y-target**
(equal world-unit displacement, in-frustum), render each, and use the closed-form **Readout injection** editor on the
passive latent to reach it. We report **readout reach** (fraction of the position gap closed) and **obs reach**
(fraction of the way to the target render, in obs space), then the **content-gen ratio = y-reach / x-reach**. `M_dxdy`
(trained on both axes) is the reference (ratio ≈ 1 expected); `M_teleport`/Baseline included for context.

In [ ]:
# [19] Content-generalization: build pure-x and pure-y targets from each edit object's pre-edit position; render them.
DISP = 2.5   # world-unit displacement along the tested axis (large, comparable to accepted training moves)
xn, xf, yn2, yf2, rad = SIM4.x_near, SIM4.x_far, SIM4.y_near, SIM4.y_far, SIM4.radius
from pim.simulator.sim import frustum_half_width
base_pos = edits.positions[:N, ef - 1, :N_OBJ, :].astype(np.float32).copy()   # pre-edit positions (both objects)
def make_axis_targets(axis):
    tp = base_pos.copy()
    for i in range(N):
        o = edit_obj[i]; x, y = base_pos[i, o]
        if axis == "x":
            xlim = float(frustum_half_width(y, SIM4)) - rad
            nx_ = x + DISP if x <= 0 else x - DISP           # move toward frustum centre to stay in-frustum
            tp[i, o] = [np.clip(nx_, -xlim, xlim), y]
        else:
            ny_ = y + DISP if y <= (yn2 + yf2) / 2 else y - DISP
            tp[i, o] = [x, np.clip(ny_, yn2 + rad, yf2 - rad)]
    return tp
AX_TP = {"x": make_axis_targets("x"), "y": make_axis_targets("y")}
AX_FLAT = {ax: torch.from_numpy(AX_TP[ax].reshape(N, N_OBJ * 2)).float().to(DEVICE) for ax in ("x", "y")}
def render_cfg1(positions):
    ri = np.zeros((N, OBS_RES), np.float32); rid = np.zeros((N, OBS_RES), np.int64)
    for i in range(N):
        sc = Scene(positions=positions[i][None], velocities=np.zeros((1, N_OBJ, 2), np.float32),
                   radii=RAD, colors=COLc, reflectivities=REFL, config=cfg1)
        _, idr, rint = render_scene(sc); ri[i] = rint[0]; rid[i] = idr[0]
    return ri, rid
BASE_RENDER, BASE_ID = render_cfg1(base_pos)                       # object k at its un-edited (pre-edit) position
AX_RENDER, AX_ID, AX_KMASK = {}, {}, {}
for ax in ("x", "y"):
    AX_RENDER[ax], AX_ID[ax] = render_cfg1(AX_TP[ax])
    km = np.zeros((N, OBS_RES), bool)
    for i in range(N):
        km[i] = (BASE_ID[i] == edit_obj[i]) | (AX_ID[ax][i] == edit_obj[i])   # obj-k rays: source ∪ axis-target
    AX_KMASK[ax] = km
print(f"axis targets built | displacement {DISP} units | x-target mean |Δpos| "
      f"{np.abs(AX_TP['x'][np.arange(N), edit_obj] - base_pos[np.arange(N), edit_obj]).sum(1).mean():.2f} | "
      f"y-target {np.abs(AX_TP['y'][np.arange(N), edit_obj] - base_pos[np.arange(N), edit_obj]).sum(1).mean():.2f} | "
      f"x k-rays {int(AX_KMASK['x'].sum())} y k-rays {int(AX_KMASK['y'].sum())}")


In [ ]:
# [20] Content-generalization: PCA-geodesic edit of the passive latent toward each axis target; robust obs-space reach.
#   readout reach: closed-form linear-probe coordinate reach (≈1 for injection — TRIVIAL, reported for context).
#   obs proj reach: obj-k obs-change projected onto the IDEAL direction (target_render − base_render), on obj-k rays;
#     = <Δedit, Δideal>/<Δideal, Δideal>, robust (bounded denominator). This is the grabbable-in-obs measure.
CG_MODELS = ["Baseline", "M_dxdy", "M_teleport", "M_axis"]
def proj_reach(o_ed0, o_base0, tgt_render, kmask):
    num = np.zeros(N); den = np.zeros(N)
    for i in range(N):
        m = kmask[i]
        if m.sum() == 0: den[i] = np.nan; continue
        de = (o_ed0[i] - o_base0[i])[m]; di = (tgt_render[i] - o_base0[i])[m]
        num[i] = float((de * di).sum()); den[i] = float((di * di).sum())
    r = num / np.where(den > 1e-9, den, np.nan)
    return float(np.nanmedian(np.clip(r, -1, 2)))
CG = {}
for n in CG_MODELS:
    P = WM[n]; row = {}
    o_base = rollout_from_flat(P["model"], P["h0"].detach().cpu().numpy(), 1)[:, 0, :]   # unsteered decode
    for ax in ("x", "y"):
        h_ed = pca_geodesic(P, P["h0"], AX_FLAT[ax])
        o_ed = rollout_from_flat(P["model"], h_ed.detach().cpu().numpy(), 1)[:, 0, :]
        r_un = (readout(P, P["h0"]) - AX_FLAT[ax]).pow(2).mean(1).sqrt()
        r_ed = (readout(P, h_ed) - AX_FLAT[ax]).pow(2).mean(1).sqrt()
        row[f"{ax}_read"] = float((1 - (r_ed / r_un.clamp_min(1e-9))).clamp(-1, 1).mean())
        row[f"{ax}_obs"] = proj_reach(o_ed, o_base, AX_RENDER[ax], AX_KMASK[ax])
    row["ratio_read"] = row["y_read"] / max(abs(row["x_read"]), 1e-6)
    row["ratio_obs"] = row["y_obs"] / row["x_obs"] if abs(row["x_obs"]) > 1e-3 else float("nan")
    CG[n] = row
print("=== Content generalization — PCA-geodesic edit of the passive latent (y never trained for M_axis) ===")
both_table(CG, [("x_read", "x readout reach", "{:.2f}"), ("y_read", "y readout reach", "{:.2f}"),
                ("x_obs", "x obs-proj reach", "{:.2f}"), ("y_obs", "y obs-proj reach", "{:.2f}"),
                ("ratio_obs", "y/x obs-proj ratio", "{:.2f}")], row_hdr="model")

fig, ax = plt.subplots(figsize=(8.5, 4.6)); w = 0.38; xs = np.arange(len(CG_MODELS))
ax.bar(xs - w / 2, [CG[n]["x_obs"] for n in CG_MODELS], w, color=OK["sky"], label="x-edit (M_axis: trained axis)")
ax.bar(xs + w / 2, [CG[n]["y_obs"] for n in CG_MODELS], w, color=OK["red"], label="y-edit (M_axis: NEVER-trained axis)")
for i, n in enumerate(CG_MODELS):
    top = max(CG[n]["x_obs"], CG[n]["y_obs"])
    ax.text(i, top + 0.015, f"y/x={CG[n]['ratio_obs']:.2f}" if not np.isnan(CG[n]['ratio_obs']) else "y/x=nan", ha="center", fontsize=8)
ax.axhline(0, color="0.5", lw=0.8)
ax.set_xticks(xs); ax.set_xticklabels([SHORT[n] for n in CG_MODELS], fontsize=9)
ax.set_ylabel("obs-projection reach on obj-k rays (fraction of ideal move)")
ax.set_title("Fig 6 — Content generalization: passive-latent geodesic edit along y (never trained) vs x (trained)")
ax.legend(fontsize=8); style_ax(ax)
fig.tight_layout(); fig.savefig(f"{OUT}/fig6_content_gen.png", dpi=130, bbox_inches="tight"); display(fig); plt.close(fig)


## Summary — does any action space individuate a selective, generalizable object handle in the passive latent?

`Baseline` = clean passive GRU · `Perturbed-passive (teleport)` = same teleport trajectories, action withheld ·
`M_dxdy` / `M_teleport` / `M_axis` = action-conditioned. **Baseline→Perturbed-passive** = perturbation-diversity;
**Perturbed-passive→M_teleport** = action-knowledge (the headline gap). All numbers are on the passive (no-op) latent,
dataset-4 held-out, with the §4 editors (an untrained write-mechanism). Interpretation lives here; body sections state
quantities only.

In [ ]:
# [21] Summary — consolidated headline numbers (all five passive models) + Fig 7 summary bars + verdict scaffolding.
fig, axes = plt.subplots(1, 2, figsize=(15.5, 4.8))
ax = axes[0]
groups = ["pos R²\n(MLP)", "vel R²\n(late MLP)", "1 - fiber resid\n(MLP g)"]
gv = {n: [pos_res[(n, "mlp")]["r2"], vel_res[(n, "late")]["mlp"]["r2"], 1 - fiber[n]["mlp_rf"]] for n in MODEL_ORDER}
x = np.arange(3); w = 0.8 / len(MODEL_ORDER)
for j, n in enumerate(MODEL_ORDER):
    off = (j - (len(MODEL_ORDER) - 1) / 2) * w
    ax.bar(x + off, gv[n], w * 0.92, color=MCOLOR[n], label=SHORT[n])
ax.set_xticks(x); ax.set_xticklabels(groups, fontsize=8.5); ax.set_ylim(0, 1.15)
ax.set_ylabel("score (0-1, higher better)"); ax.set_title("(a) passive recoverability & canonicality")
ax.legend(fontsize=7.5, ncol=2); style_ax(ax)
ax = axes[1]
bx = np.arange(len(MODEL_ORDER))
reach_v = [SCARD[n]["reach"] for n in MODEL_ORDER]; collat_v = [SCARD[n]["collat"] for n in MODEL_ORDER]
ax.bar(bx - 0.2, reach_v, 0.38, color=[MCOLOR[n] for n in MODEL_ORDER], label="reach %swap")
ax.bar(bx + 0.2, collat_v, 0.38, color="0.75", label="collateral %swap")
ax.axhline(100, color="0.3", ls="--", lw=1.1, label="true swap 100%")
for i, n in enumerate(MODEL_ORDER): ax.text(bx[i]-0.2, reach_v[i]+1.5, f"{reach_v[i]:.0f}", ha="center", fontsize=7)
ax.set_xticks(bx); ax.set_xticklabels([SHORT[n] for n in MODEL_ORDER], fontsize=8, rotation=20)
ax.set_ylabel("% of true-state swap"); ax.set_title("(b) object-handle reach vs collateral (best structural editor)")
ax.legend(fontsize=7.5); style_ax(ax)
fig.suptitle("Fig 7 — Summary across the five passive models", y=1.03, fontsize=13)
fig.tight_layout(); fig.savefig(f"{OUT}/fig7_summary.png", dpi=130, bbox_inches="tight"); display(fig); plt.close(fig)

display(Markdown("### Consolidated headline numbers (passive latent; **Pert-pass → M_teleport = action-knowledge**)"))
display(Markdown("**§1–§3 geometry / recoverability / canonicality**"))
both_table({n: dict(twonn=GEO[n]["twonn"], curv=GEO[n]["curv"], pos_mlp=pos_res[(n, "mlp")]["r2"],
                    vel_mlp=vel_res[(n, "late")]["mlp"]["r2"], fib=fiber[n]["mlp_rf"]) for n in MODEL_ORDER},
    [("twonn", "intrinsic dim", "{:.2f}"), ("curv", "curvature (deg)", "{:.1f}"), ("pos_mlp", "pos R² (MLP)", "{:.3f}"),
     ("vel_mlp", "vel R² (MLP)", "{:.3f}"), ("fib", "fiber resid (MLP)", "{:.3f}")], row_hdr="model")
display(Markdown("**§4 object-handle scorecard** (best structural editor; passive latent)"))
both_table({n: SCARD[n] for n in MODEL_ORDER},
    [("editor", "best editor", "{}"), ("reach", "reach %swap ↑", "{:.1f}"), ("collat", "collateral %swap ↓", "{:.1f}"),
     ("select", "selectivity ↑", "{:.2f}"), ("ghost", "ghost ratio ↓", "{:.3f}"), ("persist", "persistence ↑", "{:.2f}")],
    row_hdr="model")
display(Markdown("**Content generalization** (M_axis y vs x; obs-projection reach on obj-k rays; y/x ratio ≈1 ⇒ generalises)"))
both_table({n: CG[n] for n in CG_MODELS},
    [("x_obs", "x obs-proj reach", "{:.2f}"), ("y_obs", "y obs-proj reach", "{:.2f}"),
     ("ratio_obs", "y/x obs-proj ratio", "{:.2f}"), ("x_read", "x readout reach (trivial)", "{:.2f}")], row_hdr="model")

# verdict scaffolding (quantified deltas for the scratch note)
ak = {k: SCARD["M_teleport"][k] - SCARD["Perturbed-passive (teleport)"][k] for k in ("reach", "collat", "ghost", "persist")}
pd = {k: SCARD["Perturbed-passive (teleport)"][k] - SCARD["Baseline"][k] for k in ("reach", "collat", "ghost", "persist")}
print("=== VERDICT SCAFFOLDING (deltas on the object-handle scorecard) ===")
print(f"perturbation-diversity (Baseline->Pert-pass): reach {pd['reach']:+.1f}  collat {pd['collat']:+.1f}  ghost {pd['ghost']:+.3f}  persist {pd['persist']:+.2f}")
print(f"action-knowledge (Pert-pass->M_teleport):     reach {ak['reach']:+.1f}  collat {ak['collat']:+.1f}  ghost {ak['ghost']:+.3f}  persist {ak['persist']:+.2f}")
print(f"best reach any model: {max((SCARD[n]['reach'], SHORT[n]) for n in MODEL_ORDER)}  (100% = a real teleport)")
print(f"content-gen obs-proj reach x/y: M_axis x={CG['M_axis']['x_obs']:.2f} y={CG['M_axis']['y_obs']:.2f} (ratio {CG['M_axis']['ratio_obs']:.2f}) | "
      f"M_dxdy x={CG['M_dxdy']['x_obs']:.2f} y={CG['M_dxdy']['y_obs']:.2f}")
pngs = sorted(os.path.join(OUT, f) for f in os.listdir(OUT) if f.endswith(".png"))
display(Markdown("**PNG manifest:**\n\n" + "\n".join(f"- `{p}`" for p in pngs)))
print(f"{len(pngs)} PNGs in {OUT}")
